# 10 — SWR False-Solitary Validation (live GitHub API audit)  
### *Coding Alone* pipeline

**Purpose.** Our archive-based SWR counts a PR as *solitary* when it has no non-author comment events; approval-only reviews (no text) are invisible to it, and review events were excluded for coverage comparability. This notebook audits that construct against the **live GitHub API**: for a stratified sample of PRs we classified as solitary in 2018, 2020, 2022, and 2024, it fetches the full review/comment history and measures the **false-solitary rate** (PRs with real non-author human activity we missed) and, critically, whether that rate is **stable over time**. A low, stable rate converts our most-attacked metric into our best-defended one.

**Needs:** Colab Secret `GITHUB_TOKEN` (same as notebook 04). ~600 PRs × ~3 REST calls ≈ 1,800 requests, well under the 5,000/hr limit. Resumable: every PR's result is checkpointed to Drive; re-running skips finished PRs.


## Config + setup

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
BQ_DATASET="seis"; BQ_LOCATION="US"; DRIVE_BASE="/content/drive/MyDrive/ICSE_SEIS"
YEARS=[2018,2020,2022,2024]; PER_YEAR=150
import os, json, time, requests
import pandas as pd, numpy as np
from google.colab import auth, drive, userdata
from google.cloud import bigquery
auth.authenticate_user(); client=bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive'); os.makedirs(DRIVE_BASE+"/tables", exist_ok=True)
P=f"{PROJECT_ID}.{BQ_DATASET}"
TOKEN=(userdata.get('GITHUB_TOKEN') or "").strip(); assert TOKEN, "Add GITHUB_TOKEN Colab secret."
HEAD={"Authorization":f"Bearer {TOKEN}","Accept":"application/vnd.github+json"}
JSONL=f"{DRIVE_BASE}/tables/swr_validation.jsonl"
done=set()
if os.path.exists(JSONL):
    with open(JSONL) as f:
        for line in f:
            try: done.add(json.loads(line)["key"])
            except Exception: pass
print("Ready. Already audited:", len(done))


## 1. Sample solitary PRs per year from BigQuery (same construct as the paper)

In [ ]:
frames=[]
for yr in YEARS:
    frames.append(client.query(f"""
    WITH prs AS (
      SELECT repo_id, number, ANY_VALUE(actor_login) author, MIN(created_at) open_time
      FROM `{P}.frame_events_*`
      WHERE type='PullRequestEvent' AND action='opened' AND NOT is_bot AND number IS NOT NULL
        AND EXTRACT(YEAR FROM created_at)={yr}
      GROUP BY repo_id, number),
    inter AS (
      SELECT repo_id, number,
             COUNTIF(target_author IS NULL OR actor_login != target_author) ext
      FROM `{P}.frame_events_*`
      WHERE type IN ('PullRequestReviewCommentEvent','IssueCommentEvent','CommitCommentEvent')
        AND NOT is_bot AND number IS NOT NULL
      GROUP BY repo_id, number)
    SELECT {yr} AS yr, p.repo_id, p.number, p.author
    FROM prs p LEFT JOIN inter i USING(repo_id, number)
    WHERE IFNULL(i.ext,0)=0
    ORDER BY FARM_FINGERPRINT(CONCAT(CAST(p.repo_id AS STRING),'-',CAST(p.number AS STRING)))
    LIMIT {PER_YEAR}""").result().to_dataframe())
sample=pd.concat(frames,ignore_index=True)
print(sample.groupby("yr").size())


## 2. Audit each PR against the live API (resumable)
Repository is resolved by **stable numeric ID** (`GET /repositories/{id}`), so renames do not break the audit; deleted/private repos are recorded as unresolvable and reported, not silently dropped. A PR is **false-solitary** if any review or comment by a non-author human exists; approval-only reviews (empty body) are tallied separately, since those are exactly what the archive metric cannot see.

In [ ]:
def gh(url):
    for i in range(6):
        r=requests.get(url, headers=HEAD, timeout=30)
        if r.status_code==200: return r.json()
        if r.status_code in (404,410,451): return None
        if r.status_code in (403,429):
            reset=int(r.headers.get("x-ratelimit-reset",time.time()+60))
            wait=max(5,reset-time.time()+2); print(f"  rate-limited; sleep {wait:.0f}s"); time.sleep(wait); continue
        time.sleep(2**i)
    return "ERR"

def is_human(u, author):
    if not u: return False
    login=u.get("login","");
    return u.get("type")!="Bot" and not login.endswith("[bot]") and login!=author

audited=0
with open(JSONL,"a") as out:
    for _,row in sample.iterrows():
        key=f"{row.yr}-{row.repo_id}-{row.number}"
        if key in done: continue
        rec={"key":key,"yr":int(row.yr),"repo_id":int(row.repo_id),"number":int(row.number)}
        repo=gh(f"https://api.github.com/repositories/{row.repo_id}")
        if repo in (None,"ERR") or not isinstance(repo,dict):
            rec["status"]="unresolvable"
        else:
            full=repo["full_name"]
            revs=gh(f"https://api.github.com/repos/{full}/pulls/{row.number}/reviews?per_page=100")
            coms=gh(f"https://api.github.com/repos/{full}/issues/{row.number}/comments?per_page=100")
            rcoms=gh(f"https://api.github.com/repos/{full}/pulls/{row.number}/comments?per_page=100")
            if any(x=="ERR" for x in (revs,coms,rcoms)):
                rec["status"]="api_error"
            elif revs is None and coms is None and rcoms is None:
                rec["status"]="pr_gone"
            else:
                revs=revs or []; coms=coms or []; rcoms=rcoms or []
                h_rev=[r for r in revs if is_human(r.get("user"),row.author)]
                h_rev_textless=[r for r in h_rev if not (r.get("body") or "").strip()]
                h_com=[x for x in coms+rcoms if is_human(x.get("user"),row.author)]
                rec.update(status="ok",
                           n_human_reviews=len(h_rev),
                           n_approval_only=len(h_rev_textless),
                           n_human_comments=len(h_com),
                           false_solitary=int(len(h_rev)+len(h_com)>0),
                           missed_by_reviews_only=int(len(h_rev)>0 and len(h_com)==0))
        out.write(json.dumps(rec)+"\n"); out.flush(); done.add(key); audited+=1
        if audited%25==0: print(f"  audited {audited} new PRs...")
print("Audit complete. Total records:", len(done))


## 3. Summary: false-solitary rate per year, Wilson CIs, temporal stability

In [ ]:
from statsmodels.stats.proportion import proportion_confint
res=pd.read_json(JSONL, lines=True)
ok=res[res.status=="ok"].drop_duplicates("key")
rows=[]
for yr,g in ok.groupby("yr"):
    k=int(g.false_solitary.sum()); n=len(g)
    lo,hi=proportion_confint(k,n,method="wilson")
    rows.append(dict(year=yr, audited=n, false_solitary=k, rate=round(k/n,3),
                     wilson_lo=round(lo,3), wilson_hi=round(hi,3),
                     approval_only_misses=int(g.missed_by_reviews_only.sum())))
summ=pd.DataFrame(rows); summ.to_csv(f"{DRIVE_BASE}/tables/swr_validation_summary.csv",index=False)
print("=== False-solitary audit ==="); print(summ.to_string(index=False))
print("\nUnresolvable/gone/error:", res[res.status!="ok"].status.value_counts().to_dict())
# temporal stability: logistic regression of false_solitary on year
import statsmodels.formula.api as smf
ok2=ok.copy(); ok2["yr_c"]=ok2.yr-ok2.yr.mean()
m=smf.logit("false_solitary ~ yr_c", data=ok2).fit(disp=0)
print(f"\nTrend test (logit on year): coef={m.params['yr_c']:.4f}, p={m.pvalues['yr_c']:.3f}")
print("Interpretation: non-significant coef = misclassification rate is stable over time,")
print("so it cannot manufacture the SWR trend.")


## Done ✅ — paste back the summary table, the status counts, and the trend-test line.